# Notebook 04 — Case Solution Reuse (CBR Pidana Penadahan)

Notebook ini mengimplementasikan **Tahap Reuse** dari pipeline Case-Based Reasoning (CBR) untuk putusan pidana penadahan.

Notebook ini menggunakan langsung:
- `data/processed/cases.csv` (dari Notebook 02)
- `models/tfidf_vectorizer.pkl` dan `models/svm_model.pkl` (dari Notebook 03)

Tidak ada training ulang TF-IDF/SVM di notebook ini — model dimuat ulang (`joblib.load`) lalu dipakai untuk:
1. Retrieval top-K kasus mirip (cosine similarity)
2. `majority_vote()` — voting label berdasarkan jumlah suara terbanyak dari top-K
3. `weighted_vote()` — voting label berdasarkan total bobot similarity
4. `predict_outcome(query)` — menggabungkan keduanya menjadi satu prediksi solusi kasus baru

**Output notebook ini:**
- `data/results/predictions.csv`


## A. Import Library & Setup Folder

Menyiapkan library yang dibutuhkan dan memastikan folder output (`data/results`) tersedia.

In [43]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

# ======================================
# PATH PROJECT
# ======================================

# Jika notebook berada di folder notebooks/
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_PROCESSED_PATH = os.path.join(BASE_DIR, "data", "processed", "cases.csv")

MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "data", "results")

TFIDF_PATH = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
SVM_PATH = os.path.join(MODELS_DIR, "svm_model.pkl")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("========== PROJECT PATH ==========")
print("BASE_DIR            :", BASE_DIR)
print("DATA_PROCESSED_PATH :", DATA_PROCESSED_PATH)
print("MODELS_DIR          :", MODELS_DIR)
print("RESULTS_DIR         :", RESULTS_DIR)
print("TFIDF_PATH          :", TFIDF_PATH)
print("SVM_PATH            :", SVM_PATH)
print("==================================")

========== PROJECT PATH ==========
BASE_DIR            : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN
DATA_PROCESSED_PATH : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/data/processed/cases.csv
MODELS_DIR          : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/models
RESULTS_DIR         : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/data/results
TFIDF_PATH          : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/models/tfidf_vectorizer.pkl
SVM_PATH            : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/models/svm_model.pkl


## B. Load Dataset dan Model

Memuat ulang dataset hasil Notebook 02 dan model (`tfidf_vectorizer`, `svm_model`) hasil Notebook 03. Tidak ada training ulang di sini.


In [42]:
df = pd.read_csv(DATA_PROCESSED_PATH)
df["text_full"] = df["text_full"].astype(str)
df = df.reset_index(drop=True)

print("Jumlah kasus dalam case base:", len(df))
print("Distribusi label 'putusan':")
print(df["putusan"].value_counts())


Jumlah kasus dalam case base: 50
Distribusi label 'putusan':
putusan
Sedang    27
Ringan    16
Berat      7
Name: count, dtype: int64


In [5]:
tfidf_vectorizer = joblib.load(TFIDF_PATH)
svm_model = joblib.load(SVM_PATH)

print("Model berhasil dimuat:")
print("-", TFIDF_PATH)
print("-", SVM_PATH)
print()
print("Jumlah fitur TF-IDF:", len(tfidf_vectorizer.get_feature_names_out()))
print("Kelas pada SVM model:", list(svm_model.classes_))


Model berhasil dimuat:
- /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/models/tfidf_vectorizer.pkl
- /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/models/svm_model.pkl

Jumlah fitur TF-IDF: 5000
Kelas pada SVM model: ['Berat', 'Ringan', 'Sedang']


In [6]:
# TF-IDF matrix untuk seluruh case base (basis pencarian retrieval)
X_full_tfidf = tfidf_vectorizer.transform(df["text_full"])

print("Shape TF-IDF case base:", X_full_tfidf.shape)


Shape TF-IDF case base: (50, 5000)


## C. Fungsi Retrieval (Top-K)

Fungsi `retrieve(query, k=5)` menghitung cosine similarity antara query dan seluruh kasus di case base, lalu mengembalikan top-K kasus paling mirip beserta label `putusan` dan skor similarity-nya.

Fungsi ini setara dengan fungsi `retrieve()` pada Notebook 03, didefinisikan ulang secara mandiri di sini supaya Notebook 04 tidak bergantung pada eksekusi Notebook 03 sebelumnya.


In [7]:
def preprocess_query(query: str) -> str:
    """Preprocessing sederhana untuk query teks."""
    return str(query).strip().lower()


def retrieve(query: str, k: int = 5) -> pd.DataFrame:
    """
    Mengambil top-k kasus paling mirip dengan query menggunakan TF-IDF + cosine similarity.

    Returns
    -------
    pd.DataFrame dengan kolom: case_id, nomor_perkara, putusan, similarity
    diurutkan dari similarity tertinggi ke terendah.
    """
    clean_query = preprocess_query(query)
    query_vec = tfidf_vectorizer.transform([clean_query])

    sims = cosine_similarity(query_vec, X_full_tfidf).flatten()
    top_idx = np.argsort(sims)[::-1][:k]

    result = df.loc[top_idx, ["case_id", "nomor_perkara", "putusan"]].copy()
    result["similarity"] = sims[top_idx]
    result = result.reset_index(drop=True)

    return result


print("Fungsi retrieve() siap digunakan.")


Fungsi retrieve() siap digunakan.


In [8]:
# Uji cepat fungsi retrieve()
contoh_hasil = retrieve("kasus penadahan sepeda motor hasil curian", k=5)
contoh_hasil


,case_id,nomor_perkara,putusan,similarity
0,case_004,19/pid.b/2021/pn,Ringan,0.207665
1,case_026,224/pid.b/2022/pn,Ringan,0.182961
2,case_010,66/pid.b/2020/pn,Ringan,0.175649
3,case_012,74/pid.b/2021/pn,Sedang,0.166442
4,case_033,430/pid.b/2021/pn,Ringan,0.143502


## D. Majority Vote

Fungsi `majority_vote(top_cases)` menghitung label `putusan` yang paling sering muncul pada hasil top-K retrieval.

Contoh: dari 5 kasus dengan label `Sedang, Sedang, Ringan, Sedang, Berat` → hasil majority vote adalah **Sedang**.

Jika terjadi seri (tie), label dengan total similarity tertinggi pada kelompok yang seri akan dipilih sebagai tie-breaker (lebih stabil dibanding pemilihan acak).


In [9]:
def majority_vote(top_cases: pd.DataFrame) -> str:
    """
    Menentukan label putusan berdasarkan jumlah suara terbanyak (majority vote)
    dari hasil top-K retrieval.

    Parameters
    ----------
    top_cases : pd.DataFrame
        Output dari retrieve(), harus memiliki kolom 'putusan' dan 'similarity'.

    Returns
    -------
    str: label putusan hasil majority vote.
    """
    counts = top_cases["putusan"].value_counts()
    max_count = counts.max()
    kandidat = counts[counts == max_count].index.tolist()

    if len(kandidat) == 1:
        return kandidat[0]

    # Tie-breaker: pilih label dengan total similarity tertinggi di antara kandidat seri
    sim_per_label = (
        top_cases[top_cases["putusan"].isin(kandidat)]
        .groupby("putusan")["similarity"]
        .sum()
    )
    return sim_per_label.idxmax()


# Uji cepat
print("Hasil majority_vote:", majority_vote(contoh_hasil))


Hasil majority_vote: Ringan


## E. Weighted Vote

Fungsi `weighted_vote(top_cases)` menjumlahkan skor `similarity` untuk setiap label `putusan` pada hasil top-K, kemudian memilih label dengan total bobot similarity terbesar.

Pendekatan ini memberi pengaruh lebih besar pada kasus yang benar-benar mirip (similarity tinggi), bukan hanya menghitung jumlah suara seperti majority vote.


In [10]:
def weighted_vote(top_cases: pd.DataFrame) -> str:
    """
    Menentukan label putusan berdasarkan total bobot similarity (weighted vote)
    dari hasil top-K retrieval.

    Parameters
    ----------
    top_cases : pd.DataFrame
        Output dari retrieve(), harus memiliki kolom 'putusan' dan 'similarity'.

    Returns
    -------
    str: label putusan dengan total bobot similarity terbesar.
    """
    bobot_per_label = top_cases.groupby("putusan")["similarity"].sum()
    return bobot_per_label.idxmax()


# Uji cepat
print("Hasil weighted_vote:", weighted_vote(contoh_hasil))


Hasil weighted_vote: Ringan


## F. Predict Outcome

Fungsi `predict_outcome(query, k=5)` menggabungkan seluruh langkah di atas:
1. Retrieval top-K kasus mirip
2. Majority vote
3. Weighted vote

Output berupa dictionary:
```python
{
    "prediction_majority": "...",
    "prediction_weighted": "...",
    "top_cases": [...]
}
```


In [11]:
def predict_outcome(query: str, k: int = 5) -> dict:
    """
    Memprediksi solusi (label putusan) untuk kasus baru berdasarkan
    Case-Based Reasoning Reuse: retrieval top-K + majority vote + weighted vote.

    Parameters
    ----------
    query : str
        Teks kasus baru (ringkasan fakta / deskripsi kasus).
    k : int
        Jumlah kasus mirip yang diambil sebagai basis voting.

    Returns
    -------
    dict dengan keys: prediction_majority, prediction_weighted, top_cases
    """
    top_cases = retrieve(query, k=k)

    prediction_majority = majority_vote(top_cases)
    prediction_weighted = weighted_vote(top_cases)

    return {
        "prediction_majority": prediction_majority,
        "prediction_weighted": prediction_weighted,
        "top_cases": top_cases.to_dict(orient="records"),
    }


print("Fungsi predict_outcome() siap digunakan.")


Fungsi predict_outcome() siap digunakan.


In [12]:
# Uji cepat predict_outcome()
hasil_uji = predict_outcome("kasus penadahan sepeda motor hasil curian", k=5)

print("Prediction (majority):", hasil_uji["prediction_majority"])
print("Prediction (weighted):", hasil_uji["prediction_weighted"])
print()
print("Top cases:")
for c in hasil_uji["top_cases"]:
    print(c)


Prediction (majority): Ringan
Prediction (weighted): Ringan

Top cases:
{'case_id': 'case_004', 'nomor_perkara': '19/pid.b/2021/pn', 'putusan': 'Ringan', 'similarity': 0.20766505805727534}
{'case_id': 'case_026', 'nomor_perkara': '224/pid.b/2022/pn', 'putusan': 'Ringan', 'similarity': 0.18296103937336022}
{'case_id': 'case_010', 'nomor_perkara': '66/pid.b/2020/pn', 'putusan': 'Ringan', 'similarity': 0.1756486485374942}
{'case_id': 'case_012', 'nomor_perkara': '74/pid.b/2021/pn', 'putusan': 'Sedang', 'similarity': 0.16644164344664997}
{'case_id': 'case_033', 'nomor_perkara': '430/pid.b/2021/pn', 'putusan': 'Ringan', 'similarity': 0.1435023717072153}


## G. Demo Manual (5 Query Contoh)

Menjalankan `predict_outcome(query)` untuk 5 query contoh kasus penadahan, dan menampilkan hasil prediksi (majority & weighted) beserta top-5 kasus pendukungnya.


In [28]:
# ======================================
# CELL 7 - LOAD QUERIES & DEMO PREDICTION
# ======================================

# Membaca query uji dari Notebook 03
QUERIES_PATH = os.path.join(BASE_DIR, "data", "eval", "queries.json")

with open(QUERIES_PATH, "r", encoding="utf-8") as f:
    queries = json.load(f)

print(f"Jumlah query uji : {len(queries)}")
print()

demo_results = []

for item in queries:

    query_id = item["query_id"]
    query_text = item["query_text"]

    # Ground truth dari Notebook 03
    true_label = item["ground_truth"]
    ground_truth_case = item["case_id"]
    ground_truth_nomor = item["nomor_perkara"]

    # Prediksi menggunakan CBR
    hasil = predict_outcome(query_text, k=5)

    demo_results.append({
        "query_id": query_id,
        "query_text": query_text,
        "true_label": true_label,
        "ground_truth_case_id": ground_truth_case,
        "ground_truth_nomor": ground_truth_nomor,
        "prediction_majority": hasil["prediction_majority"],
        "prediction_weighted": hasil["prediction_weighted"],
        "top_cases": hasil["top_cases"]
    })

    print("=" * 100)
    print(f"Query ID            : {query_id}")
    print(f"Ground Truth Label  : {true_label}")
    print(f"Ground Truth Case   : {ground_truth_case}")
    print(f"Nomor Perkara       : {ground_truth_nomor}")
    print(f"Query               : {query_text[:150]}...")

    print("\nPrediction")
    print(f"- Majority : {hasil['prediction_majority']}")
    print(f"- Weighted : {hasil['prediction_weighted']}")

    print("\nTop-5 Similar Cases")

    for i, case in enumerate(hasil["top_cases"], start=1):
        print(
            f"{i}. "
            f"{case['case_id']} | "
            f"{case['nomor_perkara']} | "
            f"{case['putusan']} | "
            f"Similarity = {case['similarity']:.4f}"
        )

    print()

Jumlah query uji : 5

Query ID            : 1
Ground Truth Label  : Sedang
Ground Truth Case   : case_014
Nomor Perkara       : 101/pid.b/2023/pn.plgdemi
Query               : menimbang, bahwa terdakwa diajukan kepersidangan dengan dakwaansebagai berikut :bahwa ia terdakwa agus nadi als dok bin alwi, pada hari jum attanggal ...

Prediction
- Majority : Sedang
- Weighted : Sedang

Top-5 Similar Cases
1. case_014 | 101/pid.b/2023/pn.plgdemi | Sedang | Similarity = 0.4537
2. case_005 | 32/pid/2023/pt | Sedang | Similarity = 0.3440
3. case_021 | 163/pid.b/2022/pn | Sedang | Similarity = 0.2615
4. case_026 | 224/pid.b/2022/pn | Ringan | Similarity = 0.2386
5. case_024 | 191/pid.b/2021/pn | Ringan | Similarity = 0.2373

Query ID            : 2
Ground Truth Label  : Sedang
Ground Truth Case   : case_040
Nomor Perkara       : 883/pid.b/2023/pn
Query               : menimbang, bahwa terdakwa diajukan ke persidangan oleh penuntut umum didakwa berdasarkan surat dakwaan sebagai berikut : bahwa ter

## H. Simpan Hasil ke predictions.csv

Menyimpan seluruh hasil prediksi ke `data/results/predictions.csv` dengan kolom:
```text
query_id
query_text
prediction_majority
prediction_weighted
top_5_case_ids
```


In [35]:
# ======================================
# CELL 8 - SIMPAN predictions.csv
# ======================================

rows = []

for hasil in demo_results:

    top_case_ids = [
        str(case["case_id"])
        for case in hasil["top_cases"]
    ]

    rows.append({
        "query_id": hasil["query_id"],
        "query_text": hasil["query_text"],
        "true_label": hasil["true_label"],
        "prediction_majority": hasil["prediction_majority"],
        "prediction_weighted": hasil["prediction_weighted"],
        "top_5_case_ids": ";".join(top_case_ids)
    })

predictions_df = pd.DataFrame(rows)

predictions_path = os.path.join(
    RESULTS_DIR,
    "predictions.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False,
    encoding="utf-8-sig"
)

print("✅ predictions.csv berhasil disimpan")
print(predictions_path)

display(predictions_df)

✅ predictions.csv berhasil disimpan
/Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/data/results/predictions.csv


,query_id,query_text,true_label,prediction_majority,prediction_weighted,top_5_case_ids
0,1,"menimbang, bahwa terdakwa diajukan kepersidang...",Sedang,Sedang,Sedang,case_014;case_005;case_021;case_026;case_024
1,2,"menimbang, bahwa terdakwa diajukan ke persidan...",Sedang,Sedang,Sedang,case_040;case_044;case_001;case_019;case_032
2,3,menimbang bahwa putusan pengadilan tinggi jamb...,Ringan,Sedang,Sedang,case_032;case_036;case_031;case_046;case_035
3,4,menimbang bahwa putusan pengadilan tinggi band...,Sedang,Sedang,Sedang,case_046;case_032;case_035;case_037;case_036
4,5,"menimbang, bahwa terdakwa diajukan ke persidan...",Berat,Ringan,Ringan,case_018;case_024;case_033;case_038;case_035


## I. Validasi Hasil

Menampilkan:
- Total query yang diproses
- Distribusi prediksi (majority & weighted) per label (Ringan/Sedang/Berat)
- Preview `predictions.csv`


In [36]:
print("Total Query:", len(predictions_df))


Total Query: 5


In [37]:
print("Distribusi prediction_majority:")
print(predictions_df["prediction_majority"].value_counts())
print()
print("Distribusi prediction_weighted:")
print(predictions_df["prediction_weighted"].value_counts())


Distribusi prediction_majority:
prediction_majority
Sedang    4
Ringan    1
Name: count, dtype: int64

Distribusi prediction_weighted:
prediction_weighted
Sedang    4
Ringan    1
Name: count, dtype: int64


In [38]:
print("Preview predictions.csv:")
predictions_df.head()


Preview predictions.csv:


,query_id,query_text,true_label,prediction_majority,prediction_weighted,top_5_case_ids
0,1,"menimbang, bahwa terdakwa diajukan kepersidang...",Sedang,Sedang,Sedang,case_014;case_005;case_021;case_026;case_024
1,2,"menimbang, bahwa terdakwa diajukan ke persidan...",Sedang,Sedang,Sedang,case_040;case_044;case_001;case_019;case_032
2,3,menimbang bahwa putusan pengadilan tinggi jamb...,Ringan,Sedang,Sedang,case_032;case_036;case_031;case_046;case_035
3,4,menimbang bahwa putusan pengadilan tinggi band...,Sedang,Sedang,Sedang,case_046;case_032;case_035;case_037;case_036
4,5,"menimbang, bahwa terdakwa diajukan ke persidan...",Berat,Ringan,Ringan,case_018;case_024;case_033;case_038;case_035


In [39]:
predictions = pd.read_csv(PREDICTIONS_PATH)

print(predictions.head())

print("\nKolom:")
print(predictions.columns.tolist())

   query_id                                         query_text true_label  \
0         1  menimbang, bahwa terdakwa diajukan kepersidang...     Sedang   
1         2  menimbang, bahwa terdakwa diajukan ke persidan...     Sedang   
2         3  menimbang bahwa putusan pengadilan tinggi jamb...     Ringan   
3         4  menimbang bahwa putusan pengadilan tinggi band...     Sedang   
4         5  menimbang, bahwa terdakwa diajukan ke persidan...      Berat   

  prediction_majority prediction_weighted  \
0              Sedang              Sedang   
1              Sedang              Sedang   
2              Sedang              Sedang   
3              Sedang              Sedang   
4              Ringan              Ringan   

                                 top_5_case_ids  
0  case_014;case_005;case_021;case_026;case_024  
1  case_040;case_044;case_001;case_019;case_032  
2  case_032;case_036;case_031;case_046;case_035  
3  case_046;case_032;case_035;case_037;case_036  
4  case_018;case

In [40]:
# ======================================
# LOAD HASIL NOTEBOOK 04
# ======================================

import os
import pandas as pd

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

PREDICTIONS_PATH = os.path.join(
    BASE_DIR,
    "data",
    "results",
    "predictions.csv"
)

RETRIEVAL_PATH = os.path.join(
    BASE_DIR,
    "data",
    "eval",
    "retrieval_metrics.csv"
)

print("Predictions :", PREDICTIONS_PATH)
print("Retrieval   :", RETRIEVAL_PATH)

assert os.path.exists(PREDICTIONS_PATH), "predictions.csv tidak ditemukan!"
assert os.path.exists(RETRIEVAL_PATH), "retrieval_metrics.csv tidak ditemukan!"

predictions = pd.read_csv(PREDICTIONS_PATH)
retrieval_metrics = pd.read_csv(RETRIEVAL_PATH)

print("\nPredictions Shape :", predictions.shape)
print("Retrieval Shape  :", retrieval_metrics.shape)

Predictions : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/data/results/predictions.csv
Retrieval   : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/data/eval/retrieval_metrics.csv

Predictions Shape : (5, 6)
Retrieval Shape  : (1, 8)


## Ringkasan Notebook 04

- Dataset (`cases.csv`) dan model (`tfidf_vectorizer.pkl`, `svm_model.pkl`) dari Notebook 02 & 03 berhasil dimuat ulang tanpa training ulang.
- Fungsi `retrieve(query, k=5)` berhasil mengambil top-K kasus mirip menggunakan cosine similarity.
- Fungsi `majority_vote()` dan `weighted_vote()` berhasil diimplementasikan, termasuk penanganan tie-breaker pada majority vote.
- Fungsi `predict_outcome(query)` berhasil menggabungkan retrieval + voting menjadi satu hasil prediksi solusi kasus baru.
- Demo manual dengan 5 query kasus penadahan berhasil dijalankan dan dicatat.
- Hasil prediksi disimpan ke `data/results/predictions.csv` dengan kolom `query_id, query_text, prediction_majority, prediction_weighted, top_5_case_ids`.
- Validasi total query dan distribusi prediksi per label sudah ditampilkan.


